# Lecture 10C.0 — Feature Toolbox Project: Batch Setup + Dataset Splits


## Goal

In Lecture 10A/10B you **built features from scratch** (cepstrum, mel, MFCC, PLP-lite) using the manifest workflow.

In **Lecture 10C**, we do something closer to what you’ll do in research and industry:

- extract features using **toolboxes** (Praat, openSMILE),
- extract **modern embeddings** (wav2vec2 / HuBERT),
- build a **feature comparison table** and a mini benchmark.

✅ This notebook sets up a *reproducible batch pipeline* that all later notebooks reuse.


## What you’ll produce by the end of Thread 10C

Inside `EE519_L10C_Project/` you will have:

- `recordings/` (your wav files)
- `manifest.json` (what segments are used, labels, splits)
- `features/` (CSV/NPY feature dumps from each tool)
- `figures/` (plots you can paste into slides)
- `results/` (summary tables and mini leaderboard)

**Key idea:** one manifest → multiple feature extractors → one unified comparison.


## 1) Setup (shared utilities)


In [1]:
import os, json, re, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional audio playback
try:
    from IPython.display import Audio, display
    HAS_IPY_AUDIO = True
except Exception:
    HAS_IPY_AUDIO = False

# ---------------- Project paths (shared manifest workflow) ----------------
PROJECT_ROOT = Path.cwd() / "EE519_L10C_Project"
REC_DIR = PROJECT_ROOT / "recordings"
FIG_DIR = PROJECT_ROOT / "figures"
RES_DIR = PROJECT_ROOT / "results"
FEAT_DIR = PROJECT_ROOT / "features"
MANIFEST_PATH = PROJECT_ROOT / "manifest.json"

for d in [REC_DIR, FIG_DIR, RES_DIR, FEAT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def load_manifest(path=MANIFEST_PATH):
    if path.exists():
        return json.loads(path.read_text())
    return {"course":"EE519","module":"Lecture10C","created_utc":None,"clips":[],"splits":{}}

def save_manifest(manifest, path=MANIFEST_PATH):
    if manifest.get("created_utc") is None:
        manifest["created_utc"] = str(np.datetime64("now"))
    path.write_text(json.dumps(manifest, indent=2))
    print("Saved manifest:", path)

def save_fig(fig, name, dpi=150):
    out = FIG_DIR / name
    fig.savefig(out, dpi=dpi, bbox_inches="tight")
    print("Saved:", out)
    return out

import wave
def read_wav(path: Path):
    with wave.open(str(path), "rb") as wf:
        fs = wf.getframerate()
        n = wf.getnframes()
        x = np.frombuffer(wf.readframes(n), dtype=np.int16).astype(np.float32) / 32768.0
    return fs, x

def peak_normalize(x, target=0.98):
    m = np.max(np.abs(x)) + 1e-12
    return (x / m) * target

def play_audio(x, fs, label="audio"):
    if not HAS_IPY_AUDIO:
        print("(Audio playback not available)", label)
        return
    display(Audio(x, rate=fs))

def list_clips(manifest):
    for i,c in enumerate(manifest.get("clips", [])):
        print(f"[{i}] {c.get('label',''):14s}  {c.get('filename','')}  fs={c.get('fs','?')}  notes={c.get('notes','')}")

def get_segments(clip):
    # We reuse analysis_segments from earlier threads if present
    return clip.get("selections", {}).get("analysis_segments", {})

manifest = load_manifest()
print("Project root:", PROJECT_ROOT)
print("Clips:", len(manifest.get('clips', [])))
list_clips(manifest)


Project root: c:\Users\K\Documents\usc\ee519\ee519-lecture\lecture10c\EE519_L10C_Project
Clips: 12
[0] vowel_a         student10B_vowel_a.wav  fs=16000  notes=steady /a/
[1] vowel_i         student10B_vowel_i.wav  fs=16000  notes=steady /i/
[2] fricative_s     student10B_fricative_s.wav  fs=16000  notes=steady /s/
[3] sentence        student10B_sentence.wav  fs=16000  notes=short sentence
[4] vowel_a         student10B_vowel_a.wav  fs=16000  notes=steady /a/
[5] vowel_i         student10B_vowel_i.wav  fs=16000  notes=steady /i/
[6] fricative_s     student10B_fricative_s.wav  fs=16000  notes=steady /s/
[7] sentence        student10B_sentence.wav  fs=16000  notes=short sentence
[8] vowel_a         student10B_vowel_a.wav  fs=16000  notes=steady /a/
[9] vowel_i         student10B_vowel_i.wav  fs=16000  notes=steady /i/
[10] fricative_s     student10B_fricative_s.wav  fs=16000  notes=steady /s/
[11] sentence        student10B_sentence.wav  fs=16000  notes=short sentence


## 2) Import or reuse recordings + segments

### Option A (recommended)
Reuse the same recordings and segment selections from Lecture 10B:
- Copy your `EE519_L10B_Project/recordings/*.wav` into `EE519_L10C_Project/recordings/`
- Copy the clip entries from `EE519_L10B_Project/manifest.json` into this manifest

### Option B
Record fresh clips for this toolbox module.

**In this notebook** we support A by a “copy helper”.  
If you already have L10B recorded material, this saves time.


In [2]:
# ---- Helper: copy recordings and import manifest clips from Lecture 10B (if present) ----
L10B_ROOT = Path.cwd() / "EE519_L10B_Project"
L10B_MANIFEST = L10B_ROOT / "manifest.json"
L10B_REC = L10B_ROOT / "recordings"

def import_from_l10b():
    global manifest
    if not L10B_MANIFEST.exists():
        print("No L10B manifest found at:", L10B_MANIFEST)
        return
    m10b = json.loads(L10B_MANIFEST.read_text())
    clips10b = m10b.get("clips", [])
    if len(clips10b) == 0:
        print("L10B manifest has no clips.")
        return

    # copy wavs
    copied = 0
    if L10B_REC.exists():
        for c in clips10b:
            fn = c.get("filename")
            if not fn: 
                continue
            src = L10B_REC / fn
            dst = REC_DIR / fn
            if src.exists() and not dst.exists():
                dst.write_bytes(src.read_bytes())
                copied += 1

    # import clips (shallow copy)
    manifest["clips"] = clips10b
    manifest.setdefault("splits", {})
    save_manifest(manifest)
    print(f"Imported {len(clips10b)} clips; copied {copied} wav files.")

# Run this once if you want to reuse L10B data:
import_from_l10b()


No L10B manifest found at: c:\Users\K\Documents\usc\ee519\ee519-lecture\lecture10c\EE519_L10B_Project\manifest.json


### Micro-checkpoint
After importing (or recording), confirm:
- `Clips: N` prints > 0
- files exist under `EE519_L10C_Project/recordings/`


In [3]:
manifest = load_manifest()
print("Clips:", len(manifest.get("clips", [])))
list_clips(manifest)

# quick file existence check
missing = []
for c in manifest.get("clips", []):
    fn = c.get("filename","")
    if fn and not (REC_DIR/fn).exists():
        missing.append(fn)
if missing:
    print("⚠️ Missing wav files in recordings/:", missing[:10])
else:
    print("✅ All clip wav files found in recordings/.")


Clips: 12
[0] vowel_a         student10B_vowel_a.wav  fs=16000  notes=steady /a/
[1] vowel_i         student10B_vowel_i.wav  fs=16000  notes=steady /i/
[2] fricative_s     student10B_fricative_s.wav  fs=16000  notes=steady /s/
[3] sentence        student10B_sentence.wav  fs=16000  notes=short sentence
[4] vowel_a         student10B_vowel_a.wav  fs=16000  notes=steady /a/
[5] vowel_i         student10B_vowel_i.wav  fs=16000  notes=steady /i/
[6] fricative_s     student10B_fricative_s.wav  fs=16000  notes=steady /s/
[7] sentence        student10B_sentence.wav  fs=16000  notes=short sentence
[8] vowel_a         student10B_vowel_a.wav  fs=16000  notes=steady /a/
[9] vowel_i         student10B_vowel_i.wav  fs=16000  notes=steady /i/
[10] fricative_s     student10B_fricative_s.wav  fs=16000  notes=steady /s/
[11] sentence        student10B_sentence.wav  fs=16000  notes=short sentence
✅ All clip wav files found in recordings/.


## 3) Define a simple dataset split (train/test)

We’ll do a simple split at the **segment** level (not perfect, but OK for class):

- choose categories: `vowel_voiced`, `fricative_unvoiced`, `sentence_mixed`
- create a table listing every selected segment
- split into train/test with stratification

Later notebooks will load this split automatically.


In [4]:
from sklearn.model_selection import train_test_split

def infer_group(clip, seg_name):
    text = " ".join([clip.get("label",""), clip.get("notes",""), clip.get("filename",""), seg_name]).lower()
    if any(k in text for k in ["vowel", "aa", "ah", "ee", "ii", "oo", "uu"]): return "vowel_voiced"
    if any(k in text for k in ["fric", "sh", "s_", "f_", "z_", "th", "unvoiced"]): return "fricative_unvoiced"
    if any(k in text for k in ["sentence", "sent", "phrase", "reading"]): return "sentence_mixed"
    return "other"

rows = []
for ci, clip in enumerate(manifest.get("clips", [])):
    segs = get_segments(clip)
    for seg_name, sel in segs.items():
        grp = infer_group(clip, seg_name)
        rows.append({
            "clip_idx": ci,
            "segment": seg_name,
            "group": grp,
            "filename": clip.get("filename",""),
            "label": clip.get("label",""),
            "s0": int(sel.get("s0", 0)),
            "s1": int(sel.get("s1", 0)),
            "fs": int(clip.get("fs", 0) or sel.get("fs", 0) or 0),
        })
seg_df = pd.DataFrame(rows)
display(seg_df.head())

# micro-check: ensure we have at least 2 groups
print("Groups:", seg_df["group"].value_counts().to_dict())


,clip_idx,segment,group,filename,label,s0,s1,fs
0,0,vowel_mid,vowel_voiced,student10B_vowel_a.wav,vowel_a,16000,24000,16000
1,2,fricative_mid,fricative_unvoiced,student10B_fricative_s.wav,fricative_s,16000,24000,16000
2,3,seg1,fricative_unvoiced,student10B_sentence.wav,sentence,8000,19200,16000
3,3,vowel_mid,vowel_voiced,student10B_sentence.wav,sentence,8000,19200,16000
4,3,fricative_mid,fricative_unvoiced,student10B_sentence.wav,sentence,20000,28000,16000


Groups: {'fricative_unvoiced': 4, 'vowel_voiced': 2}


In [5]:
# Choose which groups to include in a simple classifier later
USE_GROUPS = ["vowel_voiced", "fricative_unvoiced", "sentence_mixed"]

df_use = seg_df[seg_df["group"].isin(USE_GROUPS)].copy()
if len(df_use) < 2 or df_use["group"].nunique() < 2:
    print("⚠️ Not enough usable segments. Add more selections in L10B_0 or rename segments with cues (vowel/fricative/sentence).")
else:
    idx = np.arange(len(df_use))
    y = df_use["group"].values
    tr, te = train_test_split(idx, test_size=0.33, random_state=0, stratify=y)

    df_use["split"] = "train"
    df_use.loc[df_use.index[te], "split"] = "test"

    # Save split table
    split_csv = RES_DIR / "L10C_segment_split.csv"
    df_use.to_csv(split_csv, index=False)
    print("Saved:", split_csv)

    # Save into manifest
    manifest.setdefault("splits", {})
    manifest["splits"]["segment_split_csv"] = str(split_csv.relative_to(PROJECT_ROOT))
    save_manifest(manifest)


Saved: c:\Users\K\Documents\usc\ee519\ee519-lecture\lecture10c\EE519_L10C_Project\results\L10C_segment_split.csv
Saved manifest: c:\Users\K\Documents\usc\ee519\ee519-lecture\lecture10c\EE519_L10C_Project\manifest.json


## 4) Reflection questions (write brief answers)

1) Why do we keep a **manifest** instead of “just running scripts”?  
2) What are the dangers of making a segment-level train/test split?  
3) What extra metadata would you add to the manifest to make experiments more reproducible?


### Answers

1. It helps us keep the original data and avoid storing unnecessary copies.
2. The train and test sets may not come from the same distribution.
3. I would add more tags about the volume/noise levels.

## What’s next
- **10C.1**: Praat features (pitch, formants, jitter/shimmer)
- **10C.2**: openSMILE eGeMAPS features
